# CadQuery para Arquitetos
## Notebook 1 — Introdução e Conceitos Básicos
### Versão Google Colab

---

### O que é o CadQuery?

**CadQuery** é uma biblioteca Python de modelagem geométrica 3D baseada em código. Diferente de softwares como Rhino ou AutoCAD, onde você modela arrastando e clicando, no CadQuery você **descreve** a geometria por meio de instruções escritas.

Essa abordagem traz vantagens importantes para arquitetos e projetistas:

- **Parametrização total**: qualquer dimensão pode ser uma variável, facilitando variações do projeto
- **Reprodutibilidade**: o código é a documentação do modelo
- **Automação**: é possível gerar dezenas de variações automaticamente
- **Integração**: o modelo pode ser conectado a planilhas, bancos de dados e outras ferramentas

O CadQuery é construído sobre o núcleo geométrico **OCCT (Open Cascade Technology)**, o mesmo motor usado pelo FreeCAD. Ele trabalha com geometria sólida real (B-Rep), o que significa que os modelos gerados têm volume, massa e propriedades físicas precisas.

---

### Visualização no Google Colab

Nesta versão do curso utilizamos o **Google Colab** como ambiente de execução. A visualização 3D interativa é feita pela biblioteca **cadquery-simpleViewer**, que renderiza os modelos diretamente nas células do notebook usando Plotly — sem necessidade de instalar nenhum software adicional.

> ⚠️ **Antes de começar**: execute a célula de instalação abaixo. Ela precisa rodar apenas uma vez por sessão. Se o ambiente do Colab for reiniciado, execute-a novamente.

---

## Instalação

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    print("Running in Colab, installing packages...")
    !pip install cadquery cadquery-simpleviewer
else:
    print("Not running in Colab, skipping package installation.")

## Importações

In [ ]:
import cadquery as cq
from cadquery_simpleviewer import show

---

## Primeiro Exemplo

Antes de entrar nos detalhes da biblioteca, veja como é simples criar e visualizar um objeto 3D com o CadQuery. O código abaixo cria uma caixa e a exibe com a função `show()`:

In [ ]:
caixa = cq.Workplane("XY").box(5, 3, 2)

show(caixa)

Em apenas duas linhas: uma cria a geometria, a outra a exibe. O gráfico gerado é interativo — você pode **orbitar**, **aproximar** e **afastar** com o mouse diretamente na célula.

Nos próximos tópicos vamos entender o que cada parte desse código significa.

---

## O Plano de Trabalho (Workplane)

Toda modelagem no CadQuery começa com um **Workplane** (plano de trabalho). Pense nele como a "superfície" sobre a qual você começa a construir, similar ao plano de referência de um software BIM.

O Workplane define:
- **Qual plano** do espaço 3D você está usando como base
- **A partir de onde** a geometria será construída

### Forma básica

Na forma mais simples, basta informar o nome do plano:

```python
cq.Workplane("XY")
```

Os três planos principais são:

| Nome | Plano | Uso comum |
|------|-------|-----------|
| `"XY"` | Horizontal (chão) | Plantas baixas, lajes |
| `"XZ"` | Frontal (fachada) | Elevações frontais |
| `"YZ"` | Lateral | Cortes laterais |

Nessa forma simples, a **origem do plano** coincide com o ponto `(0, 0, 0)` do espaço 3D.

---

### Definindo uma origem personalizada

Muitas vezes queremos começar a construir a partir de um ponto específico, e não da origem `(0, 0, 0)`. Para isso, usamos o parâmetro `origin`:

```python
cq.Workplane("XY", origin=(x, y, z))
```

Isso desloca o **ponto de partida** da construção para a coordenada escolhida. Tudo que for criado a partir desse Workplane terá sua posição calculada em relação a essa nova origem.

Veja a comparação: quatro caixas idênticas criadas com origens diferentes.

In [ ]:
# Caixa na origem padrão (0, 0, 0)
caixa_centro = cq.Workplane("XY").box(60, 60, 60)

# Caixa com origem deslocada 120mm no eixo X
caixa_direita = cq.Workplane("XY", origin=(120, 0, 0)).box(60, 60, 60)

# Caixa com origem deslocada 120mm no eixo Y
caixa_frente = cq.Workplane("XY", origin=(0, 120, 0)).box(60, 60, 60)

# Caixa com origem elevada 60mm no eixo Z
# Útil para empilhar volumes, como pavimentos de um edifício
caixa_cima = cq.Workplane("XY", origin=(0, 0, 60)).box(60, 60, 60)

show(
    [caixa_centro, caixa_direita, caixa_frente, caixa_cima],
    names=["Origem (0,0,0)", "X deslocado", "Y deslocado", "Z elevado"]
)

> 💡 **Dica**: usar `origin` no Workplane é especialmente útil quando você quer empilhar volumes (como lajes e pavimentos) ou alinhar elementos a uma grade de projeto.

---

## Formas Primitivas Básicas

O CadQuery oferece formas geométricas prontas chamadas **primitivas**. São o ponto de partida para construções mais complexas.

| Função | Descrição | Parâmetros principais |
|--------|-----------|----------------------|
| `.box(l, w, h)` | Caixa retangular | comprimento, largura, altura |
| `.cylinder(h, r)` | Cilindro | altura, raio |
| `.sphere(r)` | Esfera | raio |
| `.cone(h, r1, r2)` | Cone/Tronco de cone | altura, raio da base, raio do topo |

---

## Visualização: a função `show()`

A biblioteca **cadquery-simpleViewer** fornece a função `show()` para exibir objetos CadQuery como gráficos 3D interativos diretamente na célula.

A função aceita uma **mistura livre** de tipos na mesma chamada:

| Tipo de objeto | Como é exibido |
|----------------|----------------|
| `cq.Workplane` (sólido) | Malha 3D sólida tessellada |
| `cq.Edge` / `cq.Wire` | Linha amostrada ao longo da curva |
| `cq.Vector` / `[x, y, z]` | Marcador pontual |

### Sintaxe básica

```python
# Um único objeto
show(objeto)

# Vários objetos
show([objeto_a, objeto_b], names=["Nome A", "Nome B"])
```

### Parâmetros principais

| Parâmetro | Padrão | Descrição |
|-----------|--------|-----------|
| `objects` | — | Objeto ou lista — qualquer mix de sólidos, arestas, wires e pontos |
| `names` | `None` | Nomes para a legenda |
| `colors` | `None` | Cor de cada sólido. Consulte [Plotly CSS Colors](https://plotly.com/python/css-colors/) |
| `opacity` | `1.0` | Transparência dos sólidos — `1.0` = opaco, `0.0` = invisível |
| `visible_axes` | `"xyz"` | Eixos visíveis. `None` oculta todos. Combinações: `"x"`, `"y"`, `"z"`, `"xy"`, `"xz"`, `"yz"`, `"xyz"` |
| `z` | `None` | Cota do plano de chão. `None` = sem plano |
| `plane_color` | `"whitesmoke"` | Cor do plano de chão |
| `plane_size` | `50` | Metade do lado do plano de chão |
| `plane_opacity` | `0.8` | Transparência do plano de chão |
| `tessellation_tolerance` | `0.01` | Precisão do mesh — menor = mais fino, mais lento |
| `padding` | `0.15` | Margem adicionada ao redor da geometria |
| `points_display` | `None` | Estilo dos marcadores de ponto. Chaves: `size`, `color`, `symbol` (`"circle"`, `"square"`, `"diamond"`, `"cross"`, `"x"`), `opacity` |
| `lines_display` | `None` | Estilo das linhas (arestas e wires). Chaves: `color`, `width`, `mode` (`"lines"` ou `"lines+markers"`), `samples` (pontos amostrados por aresta, padrão `50`), `opacity` |

### Controles interativos do visualizador

Após executar `show()`, o gráfico exibe botões no topo:

- **X ● / X ○**, **Y ● / Y ○**, **Z ● / Z ○** — liga e desliga cada eixo individualmente
- **Camera** — alterna entre projeção Perspectiva e Ortográfica

Com o mouse: arraste para orbitar, scroll para zoom, arraste com botão direito para pan.

---

Vamos ver os diferentes modos de uso. Primeiro criamos os objetos:

In [ ]:
# Caixa com 80 x 80mm de base e 40mm de altura
caixa = cq.Workplane("XY").box(80, 80, 40)

# Cilindro com raio 25mm e altura 120mm
cilindro = cq.Workplane("XY").cylinder(120, 25)

### Modo técnico — eixos visíveis (padrão)

In [ ]:
show([caixa, cilindro], names=["Caixa", "Cilindro"])

### Modo limpo — sem eixos, com plano de chão

In [ ]:
show(
    [caixa, cilindro],
    names=["Caixa", "Cilindro"],
    visible_axes=None,
    z=0
)

### Com cores e plano personalizado

In [ ]:
show(
    [caixa, cilindro],
    names=["Caixa", "Cilindro"],
    colors=["lightgray", "steelblue"],
    visible_axes=None,
    z=0,
    plane_color="gainsboro",
    plane_size=200
)

> 🔎 **Resumo prático**:
> - Use `visible_axes="xyz"` (padrão) durante a modelagem para verificar dimensões e posições
> - Use `visible_axes=None` com `z=` para apresentações com visual limpo

---

## Usando Variáveis para Parametrizar

Uma das grandes vantagens de modelar em código é usar **variáveis** para as dimensões. Mudar o projeto inteiro se torna tão simples quanto alterar um único número.

In [ ]:
# Dimensões da caixa — experimente mudar e executar novamente!
caixa_comprimento = 80    # mm
caixa_largura     = 80    # mm
caixa_altura      = 40    # mm

# Dimensões do cilindro
cilindro_raio   = 25      # mm
cilindro_altura = 120     # mm

# Criando os objetos com as variáveis
caixa_param    = cq.Workplane("XY").box(caixa_comprimento, caixa_largura, caixa_altura)
cilindro_param = cq.Workplane("XY").cylinder(cilindro_altura, cilindro_raio)

show([caixa_param, cilindro_param], names=["Caixa", "Cilindro"])

---

## Movendo Objetos: `.translate()`

Para posicionar um sólido no espaço após criá-lo, usamos o método `.translate()`. Ele recebe uma **tupla** com três valores: `(x, y, z)`.

- `x` → move para a direita (positivo) ou esquerda (negativo)
- `y` → move para frente (positivo) ou trás (negativo)
- `z` → move para cima (positivo) ou baixo (negativo)

> 💡 **`origin` ou `.translate()`?** O parâmetro `origin` define o ponto de partida *antes* de criar a forma. O `.translate()` move a forma *depois* de criada. Para posicionamento simples os dois chegam ao mesmo resultado. Use o que tornar o código mais fácil de ler.

In [ ]:
# Caixa posicionada na origem
caixa_fixa = cq.Workplane("XY").box(80, 80, 40)

# Cilindro criado na origem e depois deslocado 120mm para a direita
cilindro_mov = cq.Workplane("XY").cylinder(120, 25)
cilindro_mov = cilindro_mov.translate((120, 0, 0))

show(
    [caixa_fixa, cilindro_mov],
    names=["Caixa", "Cilindro deslocado"],
    visible_axes=None,
    z=0,
    plane_size=300
)

---

## Rotacionando Objetos: `.rotate()`

Para rotacionar um sólido, usamos `.rotate()`. Ele funciona definindo um **eixo de rotação** (descrito por dois pontos) e um **ângulo em graus**.

```python
objeto.rotate(ponto_inicio_eixo, ponto_fim_eixo, angulo)
```

Para os eixos principais:

| Eixo | Ponto inicial | Ponto final | Efeito visual |
|------|--------------|-------------|---------------|
| Z | `(0,0,0)` | `(0,0,1)` | Girar em planta |
| X | `(0,0,0)` | `(1,0,0)` | Inclinar para frente/trás |
| Y | `(0,0,0)` | `(0,1,0)` | Inclinar para os lados |

In [ ]:
# Placa retangular na posição original
placa_original = cq.Workplane("XY").box(120, 20, 10)

# A mesma placa rotacionada 45° no plano horizontal (em torno do eixo Z)
placa_girada = cq.Workplane("XY").box(120, 20, 10)
placa_girada = placa_girada.rotate((0, 0, 0), (0, 0, 1), 45)

show(
    [placa_original, placa_girada],
    names=["Original", "Rotacionada 45°"]
)

---

## Escalando Objetos

O CadQuery não tem um método `.scale()` diretamente acessível no Workplane. Para aplicar escala, precisamos usar o motor geométrico OCCT que está por baixo do CadQuery.

Para deixar o código claro e simples de usar no dia a dia, vamos criar uma **função auxiliar** chamada `escalar()`. Você a define uma vez e a usa sempre que precisar.

A função recebe:
- `objeto` → o objeto CadQuery a ser escalado
- `sx` → fator de escala no eixo X
- `sy` → fator de escala no eixo Y
- `sz` → fator de escala no eixo Z

Um fator `1.0` mantém a dimensão original. Um fator `2.0` dobra. Um fator `0.5` reduz à metade.

> ⚠️ **Atenção**: a escala é aplicada em relação à **origem** `(0, 0, 0)`. É recomendado escalar *antes* de transladar.

In [ ]:
from OCP.gp import gp_GTrsf, gp_Mat
from OCP.BRepBuilderAPI import BRepBuilderAPI_GTransform

def escalar(objeto, sx, sy, sz):
    """
    Aplica escala independente nos eixos X, Y e Z a um objeto CadQuery.

    objeto : objeto CadQuery (Workplane)
    sx     : fator de escala no eixo X (1.0 = sem alteração)
    sy     : fator de escala no eixo Y (1.0 = sem alteração)
    sz     : fator de escala no eixo Z (1.0 = sem alteração)
    """
    matriz = gp_Mat(
        sx,  0,  0,
         0, sy,  0,
         0,  0, sz
    )
    transformacao = gp_GTrsf()
    transformacao.SetVectorialPart(matriz)
    construtor = BRepBuilderAPI_GTransform(objeto.val().wrapped, transformacao, True)
    resultado = cq.Workplane().add(cq.Shape(construtor.Shape()))
    return resultado

In [ ]:
import copy

esfera_original = cq.Workplane("XY").sphere(30)

esfera_uniforme = escalar(copy.copy(esfera_original), sx=2.0, sy=2.0, sz=2.0)
esfera_escala_x = escalar(copy.copy(esfera_original), sx=3.0, sy=1.0, sz=1.0)
esfera_escala_y = escalar(copy.copy(esfera_original), sx=1.0, sy=3.0, sz=1.0)
esfera_escala_z = escalar(copy.copy(esfera_original), sx=1.0, sy=1.0, sz=3.0)

esfera_uniforme = esfera_uniforme.translate(( 120, 0, 0))
esfera_escala_x = esfera_escala_x.translate(( 240, 0, 0))
esfera_escala_y = esfera_escala_y.translate(( 360, 0, 0))
esfera_escala_z = esfera_escala_z.translate(( 480, 0, 0))

show(
    [esfera_original, esfera_uniforme, esfera_escala_x, esfera_escala_y, esfera_escala_z],
    names=["Original", "Uniforme (x2)", "Escala X (x3)", "Escala Y (x3)", "Escala Z (x3)"]
)

> 🔎 **Observe**: a esfera original permanece esférica. A cópia uniforme também é esférica, porém maior. As três últimas se deformam em elipsoides — cada uma alongada em uma direção diferente.

---

## Exportando o Modelo

O CadQuery permite exportar os modelos para diferentes formatos:

| Formato | Extensão | Uso |
|---------|----------|-----|
| STEP | `.step` | Troca com outros softwares CAD (Rhino, FreeCAD, Fusion 360) |
| STL  | `.stl`  | Impressão 3D, renderização |
| DXF  | `.dxf`  | Compatível com AutoCAD |

> 💡 **No Google Colab**: os arquivos são salvos no sistema de arquivos temporário do Colab. Para baixar, clique no ícone de pasta no painel esquerdo, localize o arquivo e clique em "Baixar".

In [ ]:
modelo_exportar = cq.Workplane("XY").box(100, 80, 50)

cq.exporters.export(modelo_exportar, "meu_modelo.step")
cq.exporters.export(modelo_exportar, "meu_modelo.stl")

print("Arquivos exportados com sucesso!")
print("Acesse o painel de arquivos do Colab (ícone de pasta à esquerda) para baixá-los.")

---

## Exercício

Crie um modelo simples que represente **uma composição de três volumes arquitetônicos**: uma torre cilíndrica central e dois blocos retangulares nos lados.

**Requisitos:**
1. A torre cilíndrica deve ser visivelmente mais alta que os blocos
2. Use `origin` no Workplane para posicionar pelo menos um dos volumes
3. Use `.translate()` para posicionar pelo menos um dos volumes
4. Use variáveis para todas as dimensões
5. Exiba com `show()` usando `visible_axes=None` e `z=0`
6. **Desafio**: use `colors` para atribuir uma cor diferente a cada volume

In [ ]:
# Escreva seu código aqui



---

## Resumo

Neste notebook você aprendeu:

- O que é o CadQuery e por que é útil para arquitetos
- Como instalar o CadQuery e o cadquery-simpleViewer no Google Colab
- O conceito de **Workplane** e como definir uma **origem personalizada** com `origin=(x, y, z)`
- As formas primitivas: `box`, `cylinder`, `sphere`, `cone`
- A função **`show()`** e seus parâmetros — incluindo `visible_axes`, `z`, `points_display` e `lines_display`
- Os tipos aceitos por `show()`: sólidos, arestas (`cq.Edge`), wires (`cq.Wire`), pontos (`cq.Vector`, `[x, y, z]`)
- Como usar **variáveis** para parametrizar o modelo
- Como **posicionar** objetos com `.translate()`
- Como **rotacionar** objetos com `.rotate()`
- Como **escalar** objetos com a função auxiliar `escalar()`
- Como **exportar** modelos para STEP e STL

No próximo notebook veremos como realizar **operações booleanas** — união, subtração e interseção — para criar formas mais complexas a partir da combinação de sólidos simples.

---
*CadQuery para Arquitetos — Notebook 1 (versão Google Colab)*